In [41]:
import tensorflow as tf
import numpy as np

In [42]:
FILE_PATH = "data.csv"

In [52]:
def load_csv(csv_file: str, batch_size: int, labels: list, shuffle_size: int):
    dataset = tf.data.experimental.make_csv_dataset(
        csv_file,
        batch_size=batch_size,
        num_epochs=1,
        na_value="?",
        shuffle=True,
        shuffle_buffer_size=shuffle_size
    )
    
    def extract_labels(features):
        extracted_labels = tf.stack([features.pop(label) for label in labels], axis=-1)
        
        feature_list = []
        for feature in features.values():
            feature = tf.cast(feature, tf.float32)
            feature = tf.expand_dims(feature, axis=-1)
            feature_list.append(feature)
        
        feature_tensor = tf.concat(feature_list, axis=1)
        return feature_tensor, extracted_labels
    
    dataset = dataset.map(extract_labels)
    return dataset

In [53]:
data = load_csv(
    csv_file=FILE_PATH,
    batch_size=32,
    labels=["class_normal", "class_suspicious", "class_unknown"],
    shuffle_size=10_000
)

In [54]:
data

<_MapDataset element_spec=(TensorSpec(shape=(None, 36), dtype=tf.float32, name=None), TensorSpec(shape=(None, 3), dtype=tf.int32, name=None))>

In [55]:
train_size = 0.8
val_size = 0.1

def split_dataset(dataset, train_size, valid_size):
    lenght = 172_839
    dataset = dataset.shuffle(100_000)
    train = dataset.take(int(train_size * lenght))
    val = dataset.skip(int(train_size * lenght)).take(int(valid_size * lenght))
    test = dataset.skip(int((train_size * valid_size) * lenght))
    return train, val, test

In [56]:
train_ds, val_ds, test_ds = split_dataset(data, train_size, val_size)


In [57]:
train_ds = train_ds.cache().prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.cache().prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.cache().prefetch(tf.data.AUTOTUNE)

In [65]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(36,)),  # 36 features
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(3, activation='softmax')  # 3 saídas para classificação multiclasse
])

/home/juanvieira/local/tf/env/lib/python3.11/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [66]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 128)            │         4,736 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 15,171 (59.26 KB)

 Trainable params: 15,171 (59.26 KB)

 Non-trainable params: 0 (0.00 B)

In [68]:
model.compile(optimizer='adam',
              loss='categorical_crossentropy',  # Agora usamos categorical_crossentropy pois temos 3 labels
              metrics=['accuracy'])

In [69]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(factor=0.2, patience=3)
]

In [70]:
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks
)

Epoch 1/50


2025-03-13 10:24:09.782373: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:450] ShuffleDatasetV3:84: Filling up shuffle buffer (this may take a while): 3685 of 100000


     27/Unknown 16s 4ms/step - accuracy: 0.5346 - loss: 0.9690

2025-03-13 10:24:14.119987: I tensorflow/core/kernels/data/shuffle_dataset_op.cc:480] Shuffle buffer filled.


   5383/Unknown 33s 3ms/step - accuracy: 0.9086 - loss: 0.2195

...

5402/5402 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - accuracy: 0.9701 - loss: 0.0909 - learning_rate: 0.0010
Epoch 10/50
5402/5402 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - accuracy: 0.9723 - loss: 0.0818 - learning_rate: 0.0010
Epoch 11/50
5402/5402 ━━━━━━━━━━━━━━━━━━━━ 17s 3ms/step - accuracy: 0.9667 - loss: 0.0868 - learning_rate: 0.0010
Epoch 12/50
5402/5402 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - accuracy: 0.9484 - loss: 0.1113 - learning_rate: 0.0010
Epoch 13/50
5402/5402 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - accuracy: 0.9527 - loss: 0.1077 - learning_rate: 0.0010
Epoch 14/50
5402/5402 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - accuracy: 0.9751 - loss: 0.0769 - learning_rate: 0.0010
Epoch 15/50
5402/5402 ━━━━━━━━━━━━━━━━━━━━ 19s 3ms/step - accuracy: 0.9784 - loss: 0.0651 - learning_rate: 0.0010
Epoch 16/50
5402/5402 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - accuracy: 0.9714 - loss: 0.0793 - learning_rate: 0.0010
Epoch 17/50
  44/5402 ━━━━━━━━━━━━━━━━━━━━ 19s 4ms/step - accuracy: 0.9549 - loss: 0.0887

2025-03-13 10:29:19.417587: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]


5402/5402 ━━━━━━━━━━━━━━━━━━━━ 16s 3ms/step - accuracy: 0.9756 - loss: 0.0705 - learning_rate: 0.0010
Epoch 18/50
5402/5402 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - accuracy: 0.9795 - loss: 0.0592 - learning_rate: 0.0010
Epoch 19/50
5402/5402 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - accuracy: 0.9736 - loss: 0.0819 - learning_rate: 0.0010
Epoch 20/50
5402/5402 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - accuracy: 0.9837 - loss: 0.0560 - learning_rate: 0.0010
Epoch 21/50
5402/5402 ━━━━━━━━━━━━━━━━━━━━ 17s 3ms/step - accuracy: 0.9816 - loss: 0.0546 - learning_rate: 0.0010
Epoch 22/50
5402/5402 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - accuracy: 0.9824 - loss: 0.0518 - learning_rate: 0.0010
Epoch 23/50
5402/5402 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - accuracy: 0.9807 - loss: 0.0649 - learning_rate: 0.0010
Epoch 24/50
5402/5402 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - accuracy: 0.9808 - loss: 0.0605 - learning_rate: 0.0010
Epoch 25/50
5402/5402 ━━━━━━━━━━━━━━━━━━━━ 19s 3ms/step - accuracy: 0.9685 - loss: 0.0792 - learning

2025-03-13 10:34:08.012496: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]


5402/5402 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - accuracy: 0.9462 - loss: 0.1062 - learning_rate: 0.0010
Epoch 34/50
5402/5402 ━━━━━━━━━━━━━━━━━━━━ 16s 3ms/step - accuracy: 0.9475 - loss: 0.1049 - learning_rate: 0.0010
Epoch 35/50
5402/5402 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - accuracy: 0.9466 - loss: 0.1048 - learning_rate: 0.0010
Epoch 36/50
5402/5402 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - accuracy: 0.9470 - loss: 0.1041 - learning_rate: 0.0010
Epoch 37/50
5402/5402 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - accuracy: 0.9475 - loss: 0.1032 - learning_rate: 0.0010
Epoch 38/50
5402/5402 ━━━━━━━━━━━━━━━━━━━━ 17s 3ms/step - accuracy: 0.9473 - loss: 0.1024 - learning_rate: 0.0010
Epoch 39/50
5402/5402 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - accuracy: 0.9472 - loss: 0.1021 - learning_rate: 0.0010
Epoch 40/50
5402/5402 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - accuracy: 0.9479 - loss: 0.0998 - learning_rate: 0.0010
Epoch 41/50
5402/5402 ━━━━━━━━━━━━━━━━━━━━ 17s 3ms/step - accuracy: 0.9485 - loss: 0.0975 - learning

In [ ]:
y_hat = model.predict()

/home/juanvieira/local/tf/env/lib/python3.11/site-packages/keras/src/trainers/epoch_iterator.py:151: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


ValueError: math domain error